### Allscripts Sunrise (SCM) Dose Era Hydration

In [0]:
%sql
TRUNCATE TABLE _exponent.omop_scm.dose_era;


In [0]:
%sql
WITH unit_lookup AS (
    SELECT UPPER(TRIM(CAST(source_value AS STRING))) AS source_value_key,
           MAX(omop_concept_id) AS omop_concept_id
    FROM _exponent.omop_mapping.domain_source_to_concept
    WHERE domain_id = 'Unit'
      AND source_system = 'allscripts_scm'
      AND active_flag = TRUE
    GROUP BY UPPER(TRIM(CAST(source_value AS STRING)))
), ingredient_map AS (
    SELECT DISTINCT ds.drug_concept_id, ds.ingredient_concept_id
    FROM _exponent.omop.drug_strength ds
    JOIN _exponent.omop.concept ic
      ON ds.ingredient_concept_id = ic.concept_id
     AND ic.concept_class_id = 'Ingredient'
     AND ic.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
     AND ic.invalid_reason IS NULL
    UNION
    SELECT DISTINCT ca.descendant_concept_id AS drug_concept_id, ca.ancestor_concept_id AS ingredient_concept_id
    FROM _exponent.omop.concept_ancestor ca
    JOIN _exponent.omop.concept ic
      ON ca.ancestor_concept_id = ic.concept_id
     AND ic.concept_class_id = 'Ingredient'
     AND ic.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
     AND ic.invalid_reason IS NULL
), cteDrugTarget AS (
    SELECT
        d.drug_exposure_id,
        d.person_id,
        COALESCE(
            ingredient_map.ingredient_concept_id,
            CASE
                WHEN dc.concept_class_id = 'Ingredient'
                 AND dc.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
                 AND dc.invalid_reason IS NULL
                THEN dc.concept_id
            END
        ) AS drug_concept_id,
        COALESCE(unit_lookup.omop_concept_id, 0) AS unit_concept_id,
        CASE
            WHEN d.quantity IS NOT NULL AND d.days_supply IS NOT NULL AND d.days_supply > 0
                THEN d.quantity / CAST(d.days_supply AS DOUBLE)
            ELSE d.quantity
        END AS dose_value,
        d.drug_exposure_start_date,
        d.days_supply,
        COALESCE(
            d.drug_exposure_end_date,
            CASE WHEN d.days_supply IS NOT NULL AND d.days_supply > 0
                THEN date_add(d.drug_exposure_start_date, CAST(d.days_supply AS INT))
            END,
            date_add(d.drug_exposure_start_date, 1)
        ) AS drug_exposure_end_date
    FROM _exponent.omop_scm.drug_exposure d
    LEFT JOIN ingredient_map
      ON ingredient_map.drug_concept_id = d.drug_concept_id
    LEFT JOIN _exponent.omop.concept dc
      ON dc.concept_id = d.drug_concept_id
    LEFT JOIN unit_lookup
      ON unit_lookup.source_value_key = UPPER(TRIM(CAST(d.dose_unit_source_value AS STRING)))
    WHERE d.drug_concept_id != 0
      AND (d.days_supply IS NULL OR d.days_supply >= 0)
      AND COALESCE(
            ingredient_map.ingredient_concept_id,
            CASE
                WHEN dc.concept_class_id = 'Ingredient'
                 AND dc.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
                 AND dc.invalid_reason IS NULL
                THEN dc.concept_id
            END
          ) IS NOT NULL
), cteEndDates AS (
    SELECT person_id, drug_concept_id, unit_concept_id, dose_value, date_add(event_date, -30) AS end_date
    FROM (
        SELECT
            person_id,
            drug_concept_id,
            unit_concept_id,
            dose_value,
            event_date,
            event_type,
            MAX(start_ordinal) OVER (
                PARTITION BY person_id, drug_concept_id, unit_concept_id, dose_value
                ORDER BY event_date, event_type
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS start_ordinal,
            ROW_NUMBER() OVER (
                PARTITION BY person_id, drug_concept_id, unit_concept_id, dose_value
                ORDER BY event_date, event_type
            ) AS overall_ord
        FROM (
            SELECT
                person_id,
                drug_concept_id,
                unit_concept_id,
                dose_value,
                drug_exposure_start_date AS event_date,
                -1 AS event_type,
                ROW_NUMBER() OVER (
                    PARTITION BY person_id, drug_concept_id, unit_concept_id, dose_value
                    ORDER BY drug_exposure_start_date
                ) AS start_ordinal
            FROM cteDrugTarget
            UNION ALL
            SELECT
                person_id,
                drug_concept_id,
                unit_concept_id,
                dose_value,
                date_add(drug_exposure_end_date, 30) AS event_date,
                1 AS event_type,
                NULL AS start_ordinal
            FROM cteDrugTarget
        ) rawdata
    ) e
    WHERE (2 * e.start_ordinal) - e.overall_ord = 0
), ctoDoseEraEnds AS (
    SELECT
        dt.person_id,
        dt.drug_concept_id,
        dt.unit_concept_id,
        dt.dose_value,
        dt.drug_exposure_start_date,
        MIN(e.end_date) AS dose_era_end_date
    FROM cteDrugTarget dt
    JOIN cteEndDates e
      ON dt.person_id = e.person_id
     AND dt.drug_concept_id = e.drug_concept_id
     AND dt.unit_concept_id = e.unit_concept_id
     AND dt.dose_value <=> e.dose_value
     AND e.end_date >= dt.drug_exposure_start_date
    GROUP BY dt.person_id, dt.drug_concept_id, dt.unit_concept_id, dt.dose_value, dt.drug_exposure_start_date
)
INSERT INTO _exponent.omop_scm.dose_era (
    person_id,
    drug_concept_id,
    unit_concept_id,
    dose_value,
    dose_era_start_date,
    dose_era_end_date
)
SELECT
    person_id,
    drug_concept_id,
    unit_concept_id,
    dose_value,
    MIN(drug_exposure_start_date) AS dose_era_start_date,
    dose_era_end_date
FROM ctoDoseEraEnds
GROUP BY person_id, drug_concept_id, unit_concept_id, dose_value, dose_era_end_date;

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM _exponent.omop_scm.drug_exposure WHERE drug_concept_id <> 0 AND (days_supply IS NULL OR days_supply >= 0)) AS eligible_drug_exposure_rows,
  (SELECT COUNT(*) FROM _exponent.omop_scm.dose_era) AS dose_era_rows;
